# Fine-Tuning TinyLlama (Colab Ornek Notebook)

Bu notebook, Colab GPU ortaminda TinyLlama modelini LoRA ile kisa bir ornek veri seti uzerinde fine-tune etmeyi gosterir.

In [ ]:
!nvidia-smi
!pip -q install -U transformers datasets accelerate peft trl bitsandbytes sentencepiece

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto'
)
model.config.use_cache = False

In [ ]:
examples = [
    {
        'instruction': 'Translate to English',
        'input': 'Merhaba dunya',
        'output': 'Hello world'
    },
    {
        'instruction': 'Summarize the text',
        'input': 'LLaMA is a family of language models by Meta.',
        'output': 'LLaMA is a Meta language model family.'
    },
    {
        'instruction': 'Answer the question',
        'input': 'What is 2 + 2?',
        'output': '4'
    }
]

def format_example(x):
    return {
        'text': f"### Instruction:\n{x['instruction']}\n\n### Input:\n{x['input']}\n\n### Response:\n{x['output']}"
    }

dataset = Dataset.from_list(examples).map(format_example)
dataset

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

training_args = SFTConfig(
    output_dir='tinyllama-lora-demo',
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy='no',
    max_seq_length=512
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    peft_config=peft_config,
    processing_class=tokenizer,
    dataset_text_field='text'
)

trainer.train()

In [ ]:
prompt = '### Instruction:\nAnswer the question\n\n### Input:\nWhat is the capital of Turkiye?\n\n### Response:\n'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=64, temperature=0.7)

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))